In [1]:
from __future__ import annotations

import sys

from pathlib import Path
from typing import List, Dict, Optional
import pandas as pd
from functools import reduce




In [2]:
LANCASHIRE_COUNCILS = [
        "Blackburn with Darwen",
        "Blackpool",
        "Burnley",
        "Chorley",
        "Fylde",
        "Hyndburn",
        "Lancaster",
        "Pendle",
        "Preston",
        "Ribble Valley",
        "Rossendale",
        "South Ribble",
        "West Lancashire",
        "Wyre"
]

LANCASHIRE_COUNCILS_LA_CODES = {
        "Blackburn with Darwen" : "E06000008",
        "Blackpool" : "E06000009",
        "Burnley" : "E07000117",
        "Chorley" : "E07000118",
        "Fylde" : "E07000119",
        "Hyndburn" : "E07000120",
        "Lancaster" : "E07000121",
        "Pendle" : "E07000122",
        "Preston" : "E07000123",
        "Ribble Valley" : "E07000124",
        "Rossendale" : "E07000125",
        "South Ribble" : "E07000126",
        "West Lancashire" : "E07000127",
        "Wyre" : "E07000128"
}

# Reverse it so we can map LA_Code → LA_Name
LA_CODE_TO_NAME = {v: k for k, v in LANCASHIRE_COUNCILS_LA_CODES.items()}

In [3]:
def filter_by_columns(df: pd.DataFrame, filters: Dict[str, List]) -> pd.DataFrame:
    """
    Filters a DataFrame based on multiple columns and lists of allowed values.

    Args:
        df: Input DataFrame.
        filters: Dictionary where {column_name: [list_of_acceptable_values]}.

    Returns:
        Filtered DataFrame.
    """
    mask = pd.Series(True, index=df.index)
    for col, values in filters.items():
        if col not in df.columns:
            print(f"[WARN] Column '{col}' not found — skipping.")
            continue
        mask &= df[col].isin(values)
    return df[mask]

In [4]:
def filter_by_min_value(df: pd.DataFrame, column: str, min_value: float) -> pd.DataFrame:
    """
    Keeps only rows where df[column] >= min_value.
    Ignores non-numeric rows automatically.
    """
    # Convert to numeric in case there are text or NaN entries
    df[column] = pd.to_numeric(df[column], errors="coerce")
    return df[df[column] >= min_value]


In [ ]:
#### Accessbility to Services


####### AVERAGE HOSPITAL TRAVEL TIME #######
####### JTS #######
##### https://www.gov.uk/government/statistical-data-sets/journey-time-statistics-data-tables-jts#journey-times-to-key-services-by-local-authority-jts04

# we want to capture how easy or hard it is for a resident to reach medical services by public transport/walking.

# So we are not interested in cycling or car accessibility — those groups already have better access and are not our focus.

# we will focus on Travel time in minutes to nearest GP by PT/walk => This variable represents the average travel 
# time by public transport or walking from each LSOA to the nearest GP surgery — it’s the most relevant open proxy for general healthcare
# accessibility, and correlates well with hospital accessibility patterns (since both are healthcare destinations 
# and use the same public transport network).

In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/AvgHospitalTravelTime/jts0505.ods"
SHEET_NAME = "2019"

# 1) Keep only these columns (order preserved if they exist)
KEEP_COLUMNS = [
    "GP003", "GP005", "GP002" , "GP004", "GP101", "GP106"
]

# 2) Filter rows: keep only records where FILTER_COLUMN ∈ FILTER_VALUES
FILTER_COLUMN = "GP003"          # set to None to skip filtering
FILTER_VALUES = LANCASHIRE_COUNCILS  # leave [] or None to skip filtering

# 3) Rename columns: {"existing": "new"}
RENAME_MAP = {
    "GP101": "GP_PT_walk_mins",
    "GP106" : "GP_PT_15mins_perc",
    "GP005": "LSOA_Code",
    "GP002" : "LA_Code",
    "GP004" : "Service_users_LSOA",
    "GP003" : "LA_Name",
    
}

# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# Pick engine for ODS
read_kwargs = {"sheet_name": SHEET_NAME, "skiprows": 5}
if file_path.suffix.lower() == ".ods":
    # pip install odfpy
    read_kwargs["engine"] = "odf"

# Read sheet
df = pd.read_excel(file_path, **read_kwargs)
orig_shape = df.shape

# 1) Keep only requested columns
existing_cols = [c for c in KEEP_COLUMNS if c in df.columns]
missing_cols = [c for c in KEEP_COLUMNS if c not in df.columns]
if missing_cols:
    print(f"[WARN] Missing columns (skipped): {missing_cols}", file=sys.stderr)
if not existing_cols:
    raise ValueError("None of the requested KEEP_COLUMNS exist in the sheet.")
df = df[existing_cols]

# 2) Filter rows (Excel-style)
if FILTER_COLUMN and FILTER_VALUES:
    if FILTER_COLUMN not in df.columns:
        raise KeyError(f"Filter column '{FILTER_COLUMN}' not found in columns: {list(df.columns)}")
    df = df[df[FILTER_COLUMN].isin(FILTER_VALUES)]

# 3) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    df = df.rename(columns=RENAME_MAP)

# 4) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)

print(f"[INFO] Read: {file_path} | Sheet: '{SHEET_NAME}' | Original shape: {orig_shape}")
print(f"[INFO] Kept columns: {list(df.columns)}")
print(f"[INFO] Saved CSV: {out_path} | Final shape: {df.shape}")




In [ ]:
#### Population Vulnerability

# Group Population by Age 65+
# https://www.ons.gov.uk/census/maps/choropleth/population/age/resident-age-3a/aged-15-years-and-under/


In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/PopulationAge65+/TS007-2021-3.csv"
SHEET_NAME = "TS007-2021-3"

# 2) Filter rows: keep only records where FILTER_COLUMN ∈ FILTER_VALUES
filters = {
    "Lower tier local authorities": LANCASHIRE_COUNCILS,
}

RENAME_MAP = {
    "Lower tier local authorities": "LA_Name",
}

# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)

# 2) Filter rows (Excel-style)
df = filtered_df = filter_by_columns(df, filters)
df = filtered_df_age = filter_by_min_value(df, "Age (101 categories) Code", 65)

# 3) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    df = df.rename(columns=RENAME_MAP)

# 4) sum all above 65+
grouped_df = (
    df.groupby("LA_Name", as_index=False)["Observation"]
      .sum()
      .rename(columns={"Observation": "Total_Above 65"})
)
grouped_df["LA_Code"] = grouped_df["LA_Name"].map(LANCASHIRE_COUNCILS_LA_CODES)


# 5) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
grouped_df.to_csv(out_path, index=False)





In [ ]:
## Population Vulnerability

## Deprivation 

# https://www.ons.gov.uk/census/maps/choropleth/population/household-deprivation/hh-deprivation/household-is-deprived-in-three-dimensions?lad=E07000127

# dimensions of deprivation used in the Census 2021 UK data are Employment, education, health and disability, and household overcrowding

# we will chose populations who are deprived in at least 3 dimensions

In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/Deprivation/TS011-2021-6.csv"
SHEET_NAME = "TS011-2021-6.csv"

# 2) Filter rows: keep only records where FILTER_COLUMN ∈ FILTER_VALUES
FILTER_COLUMN = "Lower tier local authorities"          # set to None to skip filtering
FILTER_VALUES = LANCASHIRE_COUNCILS  # leave [] or None to skip filtering


filters = {
    "Lower tier local authorities": LANCASHIRE_COUNCILS,
    "Household deprivation (6 categories) Code": [4]
}

RENAME_MAP = {
    "Lower tier local authorities": "LA_Name",
    "Lower tier local authorities Code" : "LA_Code"
}


# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)

# 2) Filter rows (Excel-style)
df = filtered_df = filter_by_columns(df, filters)

# 3) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    df = df.rename(columns=RENAME_MAP)

# 4) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)


In [ ]:
## Population Vulnerability

## Disability

# https://www.ons.gov.uk/census/maps/choropleth/health/disability-age-standardised/disability-4a/disabled-under-the-equality-act-day-to-day-activities-limited-a-lot

# we will chose populations within Limited a lot + Limited a little

In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/Disability/TS038ASP-2021-2.csv"
SHEET_NAME = "TS038ASP-2021-2.csv"

filters = {
    "Lower Tier Local Authorities": LANCASHIRE_COUNCILS,
    "Disability - Equality act disabled (4 categories) Code": [1,2]
}

RENAME_MAP = {
    "Lower Tier Local Authorities": "LA_Name"
}


# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)

# 2) Filter rows (Excel-style)
df = filtered_df = filter_by_columns(df, filters)

# 3) sum all limited a lot and limited a little
grouped_df = (
    df.groupby("Lower Tier Local Authorities", as_index=False)["Observation"]
      .sum()
      .rename(columns={"Observation": "Total_Disablities"})
)

# 4) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    grouped_df = grouped_df.rename(columns=RENAME_MAP)

grouped_df["LA_Code"] = grouped_df["LA_Name"].map(LANCASHIRE_COUNCILS_LA_CODES)

# 5) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
grouped_df.to_csv(out_path, index=False)

In [ ]:
## % of Households with No Car

# https://www.ons.gov.uk/census/maps/choropleth/housing/number-of-cars-or-vans/number-of-cars-3a/no-cars-or-vans-in-household/

# we will chose populations with no cars or vans in household


In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/NoCarOrVan/TS045-2021-4.csv"
SHEET_NAME = "TS045-2021-4.csv"

filters = {
    "Lower tier local authorities": LANCASHIRE_COUNCILS,
    "Car or van availability (5 categories) Code": [0]
}

RENAME_MAP = {
    "Lower tier local authorities": "LA_Name",
    "Lower tier local authorities Code" : "LA_Code"
}


# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)

# 2) Filter rows (Excel-style)
df = filtered_df = filter_by_columns(df, filters)

# 3) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    df = df.rename(columns=RENAME_MAP)

# 4) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)





In [ ]:
## Index of multiple deprivation
# Source: https://imd-by-geo.opendatacommunities.org/imd/2019/dimension?area=http://opendatacommunities.org/id/geography/administration/cty/E10000017

## 7 domains within : Income Deprivation, Employment Deprivation, Education, Skills and Training Deprivation, Health Deprivation and Disability
# Crime, Barriers to Housing and Services, Living Environment Deprivation

# Income Deprivation Affecting Children Index (IDACI) : Measures deprivation specifically among children aged 0-15
# Income Deprivation Affecting Older People Index (IDAOPI): Focuses on deprivation among those aged 60 and over


# Rank : Each LSOA is assigned a rank (from 1 to about 32,844, the total number of LSOAs in England) for the IMD and each domain, 
#       where Rank 1 is the most deprived and higher numbers are less deprived.


# Decile : LSOAs are split into ten groups—deciles—based on their rank. Decile 1 is the most deprived 10% nationally
#          and Decile 10 is the least deprived 10%.




In [ ]:
## For our transport poverty index
## We will chose Health and Disability Decile => captures Mobility and NEPT correlation
## Barriers to Housing & Services Decile => captures Distance and geographic isolation
## IDAOPI Decile => Low-income elderly (links to car ownership & access)



In [ ]:


# --- EDIT THESE VARIABLES ---
INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/IndexofMultipleDeprivation/deprivation-data-2019.csv"
SHEET_NAME = "deprivation-data-2019.csv"

filters = {
    "Postcode Status": ["Live"]
}

# 1) Keep only these columns (order preserved if they exist)
KEEP_COLUMNS = [
    "LSOA Code", "LSOA Name", "Health and Disability Decile", "Barriers to Housing & Services Decile", "IDAOPI Decile"
]

# Rename columns: {"existing": "new"}
RENAME_MAP = {
    "LSOA Code": "LSOA_Code"
}

# 4) Output CSV path (None -> auto next to input with suffix "_processed.csv")
OUTPUT_CSV = None
# --- END OF EDITS ---


file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)

# 2) Filter rows (Excel-style)
df = filtered_df = filter_by_columns(df, filters)

# 3) Keep only requested columns
existing_cols = [c for c in KEEP_COLUMNS if c in df.columns]
missing_cols = [c for c in KEEP_COLUMNS if c not in df.columns]
if missing_cols:
    print(f"[WARN] Missing columns (skipped): {missing_cols}", file=sys.stderr)
if not existing_cols:
    raise ValueError("None of the requested KEEP_COLUMNS exist in the sheet.")
df = df[existing_cols]

# 4) Rename columns
if RENAME_MAP:
    unknown_keys = [k for k in RENAME_MAP if k not in df.columns]
    if unknown_keys:
        print(f"[WARN] Rename keys not present (ignored): {unknown_keys}", file=sys.stderr)
    df = df.rename(columns=RENAME_MAP)

# 5) Save CSV
if OUTPUT_CSV is None:
    OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)





In [33]:
# Combine All IMD SubDomains


INPUT_FILE = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/IndexofMultipleDeprivation/MultipleDeprivation_LA.csv"

file_path = Path(INPUT_FILE)
if not file_path.exists():
    raise FileNotFoundError(f"Input file not found: {file_path}")

# --- Step 1: Read depending on file type ---

if file_path.suffix.lower() == ".csv":
    df = pd.read_csv(file_path)


# normalise decile: 1 = high risk (deprived) → 1.0, 10 = low risk → 0.0
def decile_to_risk(decile_series):
    s = decile_series.astype(float)
    return (10 - s) / 9


df["HealthDisability_norm"] = decile_to_risk(df["Health and Disability Decile"])
df["HousingServices_norm"] = decile_to_risk(df["Barriers to Housing & Services Decile"])
df["IDAOPI_norm"] = decile_to_risk(df["IDAOPI Decile"])

df["DeprivationCombined_norm"] = (
    df["HealthDisability_norm"] +
    df["HousingServices_norm"] +
    df["IDAOPI_norm"]
) / 3

OUTPUT_CSV = str(file_path.with_name(f"{file_path.stem}_processed.csv"))
out_path = Path(OUTPUT_CSV)
out_path.parent.mkdir(parents=True, exist_ok=True)
df.to_csv(out_path, index=False)


In [ ]:
############################ Add LA_NAME & LA_CODE column to Multiple Deprivation File.csv


In [ ]:

# ---------------------------------------------------
# 1. Path to your CSV file (the one that ONLY has LSOA Name)
# ---------------------------------------------------
file_in = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/IndexofMultipleDeprivation/MultipleDeprivation.csv"
file_out = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/IndexofMultipleDeprivation/MultipleDeprivation_LA.csv"

df = pd.read_csv(file_in)



# ---------------------------------------------------
# 3. Function to extract LA_Name from "LSOA Name"
# ---------------------------------------------------
def extract_la_name(lsoa_name: str) -> str | None:
    """
    Extract the LA short name from LSOA Name using two rules:
    1. Full LA name match at the start (e.g., 'West Lancashire 004A')
    2. If no match: take substring before first space (e.g., 'Blackpool 001A' → 'Blackpool')
    """
    if pd.isna(lsoa_name):
        return None

    s = str(lsoa_name).strip()

    # Rule 1: Does LSOA start with the full LA name?
    for full_la_name in LANCASHIRE_COUNCILS_LA_CODES.keys():
        if s.startswith(full_la_name + " ") or s == full_la_name:
            return full_la_name

    # Rule 2: fallback → first token before space
    return s.split(" ")[0]


# ---------------------------------------------------
# 4. Create LA_Name and LA_Code columns
# ---------------------------------------------------
df["LA_Name"] = df["LSOA Name"].apply(extract_la_name)

df["LA_Code"] = df["LA_Name"].map(LANCASHIRE_COUNCILS_LA_CODES)

# ---------------------------------------------------
# 5. Optional: Print rows that did NOT map
# ---------------------------------------------------
unmatched = df[df["LA_Code"].isna()][["LSOA Name", "LA_Name"]].drop_duplicates()

if not unmatched.empty:
    print("\n⚠️ WARNING: Some LA names did not map to LA_Code:")
    print(unmatched)

# ---------------------------------------------------
# 6. Save updated file
# ---------------------------------------------------
df.to_csv(file_out, index=False)

print("\n✅ Done! New columns LA_Name and LA_Code added.")


In [ ]:
############################ Add LA_NAME & LA_CODE column to Bus Frequency.csv


In [20]:

# ---------------------------------------------------
# 1. Path to your CSV file (the one that ONLY has LSOA Name)
# ---------------------------------------------------
file_in = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored_qc.csv"
file_out = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored_qc_LA.csv"

df = pd.read_csv(file_in)



# ---------------------------------------------------
# 3. Function to extract LA_Name from "LSOA Name"
# ---------------------------------------------------
def extract_la_name(lsoa_name: str) -> str | None:
    """
    Extract the LA short name from LSOA Name using two rules:
    1. Full LA name match at the start (e.g., 'West Lancashire 004A')
    2. If no match: take substring before first space (e.g., 'Blackpool 001A' → 'Blackpool')
    """
    if pd.isna(lsoa_name):
        return None

    s = str(lsoa_name).strip()

    # Rule 1: Does LSOA start with the full LA name?
    for full_la_name in LANCASHIRE_COUNCILS_LA_CODES.keys():
        if s.startswith(full_la_name + " ") or s == full_la_name:
            return full_la_name

    # Rule 2: fallback → first token before space
    return s.split(" ")[0]


# ---------------------------------------------------
# 4. Create LA_Name and LA_Code columns
# ---------------------------------------------------
df["LA_Name"] = df["LADNM"].apply(extract_la_name)

df["LA_Code"] = df["LA_Name"].map(LANCASHIRE_COUNCILS_LA_CODES)

# ---------------------------------------------------
# 5. Optional: Print rows that did NOT map
# ---------------------------------------------------
unmatched = df[df["LA_Code"].isna()][["LADNM", "LA_Name"]].drop_duplicates()

if not unmatched.empty:
    print("\n⚠️ WARNING: Some LA names did not map to LA_Code:")
    print(unmatched)


# ---------------------------------------------------
# 5. Keep only Lancashire Records
# ---------------------------------------------------

df = df[df["LA_Name"].isin(LANCASHIRE_COUNCILS)]


# ---------------------------------------------------
# 6. Save updated file
# ---------------------------------------------------
df.to_csv(file_out, index=False)

print("\n✅ Done! New columns LA_Name and LA_Code added.")



⚠️ WARNING: Some LA names did not map to LA_Code:
  LADNM LA_Name
0   NaN    None

✅ Done! New columns LA_Name and LA_Code added.


In [38]:
# ##### MASTER Table for DEMAND


# --- Hardcode your CSV file paths here ---
file1 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/IndexofMultipleDeprivation/MultipleDeprivation_LA_processed.csv"
file2 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/AvgHospitalTravelTime/AvgHospitalTravelTime.csv"
file3 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/PopulationAge65/PopAbove65.csv"
file4 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/Deprivation/Deprivation.csv"
file5 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/Disability/Disability.csv"
file6 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/NoCarOrVan/NoCarOrVan.csv"

# Read all CSVs into dataframes
df1 = pd.read_csv(file1) # has LA_Code + LA_Name
df2 = pd.read_csv(file2)
df3 = pd.read_csv(file3)
df4 = pd.read_csv(file4)
df5 = pd.read_csv(file5)
df6 = pd.read_csv(file6)


dfs = [df1, df2, df3, df4, df5, df6]

# 2. --- Helper: collapse *_x / *_y into a single clean key column ---
def fix_keys(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # ---- LA_Code ----
    if "LA_Code" not in df.columns:
        # find any LA_Code*, e.g. LA_Code_x, LA_Code_y
        la_cols = [c for c in df.columns if c.startswith("LA_Code")]
        if la_cols:
            # build one LA_Code taking first non-null across the suffixed columns
            df["LA_Code"] = df[la_cols].bfill(axis=1).iloc[:, 0]
            df.drop(columns=la_cols, inplace=True)

    # ---- LSOA_Code ----
    if "LSOA_Code" not in df.columns:
        lsoa_cols = [c for c in df.columns if c.startswith("LSOA_Code")]
        if lsoa_cols:
            df["LSOA_Code"] = df[lsoa_cols].bfill(axis=1).iloc[:, 0]
            df.drop(columns=lsoa_cols, inplace=True)

    return df


# 3. --- Optional: only keep LA_Name from the first df to avoid duplicates ---
DUPLICATE_NON_KEY_COLS = ["LA_Name"]  # extend if needed

for col in DUPLICATE_NON_KEY_COLS:
    for i in range(1, len(dfs)):  # keep col in dfs[0], drop from dfs[1:]
        if col in dfs[i].columns:
            dfs[i] = dfs[i].drop(columns=[col])


# 4. --- Smart merge: prefer LSOA_Code, else LA_Code ---
def smart_merge(left: pd.DataFrame, right: pd.DataFrame, label: str = "") -> pd.DataFrame:
    # Make sure key columns are in clean form on BOTH sides
    left = fix_keys(left)
    right = fix_keys(right)

    if "LSOA_Code" in left.columns and "LSOA_Code" in right.columns:
        key = "LSOA_Code"
    elif "LA_Code" in left.columns and "LA_Code" in right.columns:
        key = "LA_Code"
        # if right is LA-level, dedupe to avoid many-to-many explosion
        right = right.drop_duplicates(subset=["LA_Code"])
    else:
        # Helpful debugging if something is still off
        raise ValueError(
            f"No common key (LSOA_Code or LA_Code) between merged df and {label}.\n"
            f"Left cols: {list(left.columns)}\nRight cols: {list(right.columns)}"
        )

    print(f"Merging on {key} with {label}")
    return pd.merge(left, right, on=key, how="outer")


# 5. --- Merge all dataframes sequentially ---
merged = dfs[0]
for idx, df in enumerate(dfs[1:], start=2):
    merged = smart_merge(merged, df, label=f"df{idx}")

# Final clean-up: ensure keys are in canonical form
merged = fix_keys(merged)


# Fill missing LA_Name using LA_Code mapping
merged["LA_Name"] = merged["LA_Name"].fillna(
    merged["LA_Code"].map(LA_CODE_TO_NAME)
)

# 6. --- Save result ---
merged.to_csv("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/merged_output.csv", index=False)
print("Final merged shape:", merged.shape)

Merging on LSOA_Code with df2
Merging on LA_Code with df3
Merging on LA_Code with df4
Merging on LA_Code with df5
Merging on LA_Code with df6
Final merged shape: (945, 28)


In [39]:
###### Combine all (Population + Bus Frequency)

In [40]:
##### MASTER 


# --- Hardcode your CSV file paths here ---
file7 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/merged_output.csv"
file8 = "/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/TransportSupply/Timetable/Output/lsoa_bus_frequency_scored_qc_LA.csv"

# Read all CSVs into dataframes
df7 = pd.read_csv(file7) # has LA_Code + LA_Name
df8 = pd.read_csv(file8)


# ---------------------------------------------------
# 6. Merge all six on LA_Code
# ---------------------------------------------------

# inner join
merged_df_new = pd.merge(df7, df8, on='LSOA_Code', how='inner')

# Save output
merged_df_new.to_csv("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/LSOA_master.csv", index=False)

print("Merged shape:", merged_df_new.shape)


Merged shape: (896, 39)


In [37]:
###### 
##### Normalise Variables
###########




In [22]:
import pandas as pd

# === 1. Load your master LSOA file ===
# Change the path/filename if needed
df = pd.read_csv("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/LSOA_master.csv")

# EXPECTED COLUMNS (rename in code if yours differ):
# 'LSOA_Code'              -> LSOA identifier
# 'Total_Above_65'             -> population aged 75+
# '%_Disablities'              -> share of residents with long-term health problem / disability
# '%_with__No_car'                 -> share of households with no car/van
# 'DeprivationCombined_norm'             -> 1 = most deprived, 10 = least deprived
# 'GP_PT_walk_mins'      -> average public transport time to hospital (minutes)
# 'BusFreqScore'           -> higher value = better bus service (e.g. more buses/hour)

# === 2. Helper function for min–max normalisation ===
def min_max_normalise(series):
    s = series.astype(float)
    s_min = s.min()
    s_max = s.max()
    if pd.isna(s_min) or pd.isna(s_max):
        # all NaNs
        return pd.Series([pd.NA] * len(s), index=s.index, dtype="float64")
    if s_max == s_min:
        # avoid division by zero – everything identical, give 0 risk
        return pd.Series(0.0, index=s.index)
    return (s - s_min) / (s_max - s_min)


# === 3. Normalise each variable in the "risk" direction ===

# More elderly = more risk
df["Pop75_norm"] = min_max_normalise(df["Total_Above_65"])

# Higher % disabled = more risk
df["Disabled_norm"] = min_max_normalise(df["%_Disablities"])

# Higher % no car = more risk
df["NoCar_norm"] = min_max_normalise(df["%_with__No_car"])

# IMD: decile 1 = most deprived, 10 = least deprived
# Convert to a 0–1 risk score where 1 = most deprived
df["IMD_norm"] = (10 - df["DeprivationCombined_norm"].astype(float)) / 9.0
# (optional) clip in case of weird values
df["IMD_norm"] = df["IMD_norm"].clip(lower=0, upper=1)

# Longer travel time to hospital = more risk
df["HospTravel_norm"] = min_max_normalise(df["GP_PT_walk_mins"])

# Bus frequency: higher freq = LESS risk
# Step 1: normalise frequency (0 = lowest freq, 1 = highest freq)
freq_norm = min_max_normalise(df["BusFreqScore"])
# Step 2: invert so 1 = worst (fewest buses), 0 = best
df["BusFreqRisk_norm"] = 1 - freq_norm




# # === 4. Compute Transport Poverty Index (TPI) ===
# # You can tweak these weights if you like
# w_pop75     = 0.20
# w_disabled  = 0.20
# w_nocar     = 0.15
# w_imd       = 0.15
# w_hosp      = 0.15
# w_busrisk   = 0.15

# df["TransportPovertyIndex"] = (
#     w_pop75    * df["Pop75_norm"] +
#     w_disabled * df["Disabled_norm"] +
#     w_nocar    * df["NoCar_norm"] +
#     w_imd      * df["IMD_norm"] +
#     w_hosp     * df["HospTravel_norm"] +
#     w_busrisk  * df["BusFreqRisk_norm"]
# )


norm_columns = [
    "Pop75_norm",
    "Disabled_norm",
    "NoCar_norm",
    "DeprivationCombined_norm",   # replaces IMD_norm
    "HospTravel_norm",
    "BusFreqRisk_norm"
]

# Row-wise mean of the six normalised variables
df["TransportPovertyIndex"] = df[norm_columns].mean(axis=1)



# === 2. Sort by Transport Poverty Index (highest = worst) ===
df_sorted = df.sort_values(by="TransportPovertyIndex", ascending=False)

# === 3. Select top 10 LSOAs ===
top30 = df_sorted.head(30)


# If you have parent district column (optional):
if "LADNM" in df.columns:
    print("\nWith Districts:\n")
    print(top30[["LSOA_Code", "LADNM", "TransportPovertyIndex"]])

# === 6. Save to new CSV ===
df.to_csv("/Users/shrankhlagarg/Desktop/Michaelmas/CVP/Transport_Poverty_Lancashire/Data/Master_LSOA_with_TPI.csv", index=False)
print("Saved file: Master_LSOA_with_TPI.csv")



With Districts:

     LSOA_Code      LADNM  TransportPovertyIndex
596  E01025293    Preston               0.857460
589  E01025286    Preston               0.840317
565  E01025260    Preston               0.838942
569  E01025264    Preston               0.837566
114  E01012693  Blackpool               0.829786
615  E01025313    Preston               0.823280
134  E01012713  Blackpool               0.819471
135  E01012714  Blackpool               0.816086
167  E01012747  Blackpool               0.814401
88   E01012666  Blackpool               0.813902
133  E01012712  Blackpool               0.811223
172  E01012752  Blackpool               0.811215
174  E01012754  Blackpool               0.811123
139  E01012718  Blackpool               0.810750
89   E01012667  Blackpool               0.810479
586  E01025283    Preston               0.810265
111  E01012689  Blackpool               0.809132
557  E01025252    Preston               0.807513
107  E01012685  Blackpool               0.807454
10